# **Question 4.8.1**
**Apart from the arrangement symmetry of multilayer perceptrons, are there other scenarios in which neural networks might exhibit symmetry that needs to be broken?**

**1. Temporal Symmetry in Recurrent Neural Networks (RNNs)**<br><br>
RNNs share the same set of recurrent weights across different time steps. If initialized improperly, this structure may cause the network to treat all time steps equally, making it unable to distinguish the relative importance of information at different times.<br>
**Methods for breaking this:** By using orthogonal initialization or identity matrix initialization, a slight asymmetric perturbation is introduced to allow the hidden states at different time steps to diverge, thereby enabling the model to learn meaningful temporal dependencies.<br><br>

**2. Symmetry of Encoder-Decoder Weights in Autoencoders**<br><br>
When the encoder and decoder of an autoencoder use symmetric, shared weights (i.e., the decoder is the transpose of the encoder), the network tends to directly learn an identity mapping, thereby losing its ability to extract features.<br>
**Methods for breaking this:** Introducing an asymmetric regularization term, using different architectures or initializations for the encoder and decoder, or applying non-uniform $L_2$ regularization to force the network to learn meaningful compressed representations rather than simply replicating the input.<br><br>

**3. Response Symmetry Caused by Symmetric Activation Functions**<br><br>
When using activation functions that are symmetric about the origin (such as Tanh or Sine), hidden units may produce symmetric response patterns, leading to slow gradient updates or convergence stagnation.<br>
**Methods for breaking this:** Switch to an asymmetric activation function (such as ReLU or Leaky ReLU). ReLU’s one-sided saturation property naturally breaks the symmetry among inputs, promoting the specialization of neurons to specific features.<br><br>

**4. Filter Symmetry in Convolutional Neural Networks (CNNs)**<br><br>
In CNNs, if different filters in the same layer have completely symmetric initial states, they will learn redundant feature detectors, wasting network capacity.<br>
**Methods for breaking this:** Initialize them randomly, ensuring that differences exist between filters from the outset, so that they specialize in detecting different low-level features (such as edges and textures).<br><br>

# **Question 4.8.2**
**Can we initialize all the weight parameters in linear regression or softmax regression to the same value?**<br><br>

Yes. **The key difference is that single-layer linear regression and softmax regression lack permutation symmetry among hidden units.**<br>
* Linear regression: Each weight $w_j$ is associated with a different input feature $x_j$. Even if all $w_j$ are initialized to the same value (such as 0), the gradient
$$\frac{\partial{L}}{\partial{w_j}}=\frac{2}{n}\sum\limits_{i}x_{ij}(\hat{y}_{i}-y_{i})$$
This typically varies depending on the feature columns, so it will update normally. Initializing with all zeros is safe.
* Softmax Regression: If the weights and biases for all classes are initialized to the same value, the initial output is uniformly distributed between 1/K and 1/K. However, during backpropagation, the gradient for class $k$ is
$$\frac{1}{n}\sum\limits_{i}(p_{i,k}-1[y_{i}=k])x_{i}$$
Different categories receive different gradients based on their labels and begin to diverge after the first update. Therefore, initializing them with all zeros or identical values will not result in permanent symmetry.<br><br>
Summary: **In linear regression and softmax regression, all weights can be initialized to the same value (typically 0) without causing the model to become untrainable; in practice, random initialization is merely a common convention, not a requirement for breaking symmetry. The only exception is when the data itself is completely degenerate (e.g., all feature columns are identical), but that is a data issue rather than an initialization issue.**

# **Question 4.8.3**
**Look up the analytical bounds for the eigenvalues of the product of two matrices in the relevant literature. What insights does this provide for ensuring that the gradient conditions are appropriate?**<br><br>

1. **Initialization should aim for “spectral norm $\approx$ 1”**<br>
**Since the gradient norm is governed by $\prod \sigma_{max}(J_j)$, if the singular values of each layer are systematically less than 1, the gradient decays exponentially with depth (and eventually vanishes); if they are systematically greater than 1, it grows exponentially (and becomes explosive).** Therefore, a core principle of initialization is to ensure that the singular values of the weight matrices in each layer are concentrated as closely as possible around 1 (e.g., orthogonal initialization, spectral normalization), thereby preventing the spectral norm of the product from rapidly deviating from 1 along the depth dimension.<br><br>
2. **The condition number is a more precise measure than the spectral norm**<br>
Simply constraining the spectral norm (maximum singular value) to 1 does not guarantee that the minimum singular value will also be close to 1. If the minimum singular value is too small, even if the maximum singular value is 1, the condition number will still be large, and the gradient may vanish or remain sensitive to perturbations in certain directions. **Therefore, the reason orthogonal initialization (where all singular values are 1) is superior to simple spectral norm scaling is precisely because it simultaneously controls both the maximum and minimum singular values, resulting in a condition number of 1**.<br><br>
3. **The derivative of the activation function is also a “factor” and cannot be ignored**<br>
The bounds for gradient multiplication also include the spectral norm of the activation function’s derivatives $||diag(h')||$. Since the derivatives of the sigmoid and tanh functions approach 0 in the saturated region, they act as additional contraction factors that suppress the spectrum of the product, thereby exacerbating gradient vanishing. The derivative of ReLU is 1 in the activation region and 0 in the non-activation region; while this avoids saturation-induced contraction, it also alters the effective spectrum due to stochastic zeroing. Therefore, **choosing an activation function is equivalent to selecting a spectral contraction mode, which must be considered in conjunction with weight initialization**.<br><br>
4. **Early Diagnosis of “Pathological Conditions”**<br>
Analytical bounds on products provide a way to perform a priori checks: if the condition numbers of the weight matrices in two layers are both known to be large, then the condition number of their product will deteriorate in the worst case, and the convergence of gradient descent at that depth will be compromised. This suggests that during training, we can identify the source of gradient deterioration by monitoring the condition numbers of the weights (or Jacobians) at each layer, rather than simply observing a single scalar measure such as the gradient norm..<br><br>

# **Question 4.8.4**
**If we know that certain terms are divergent, can we correct them afterward?**<br><br>

Partial corrections can be made after the fact, but it is important to distinguish between two types of “divergence”:<br><br>
* **Divergence at the numerical/training level (gradient explosion, overflow, NaN)**: These issues can be remedied after the fact, such as through gradient clipping, learning rate reduction, weight normalization/spectral normalization, residual connections, LayerNorm, or rolling back to the most recent checkpoint and adjusting hyperparameters. However, once parameters have overflowed or the optimization trajectory has been disrupted, it is usually impossible to “fix it on the spot”; the only option is to roll back and retrain.
* **Divergence in the sense of analysis (e.g., systematic deviation of $\prod \sigma_{max}(J_j)$ from 1)**: It cannot be corrected by modifying the inequality itself after the fact. The product norm is determined by its individual factors; therefore, the factors must be altered—through initialization (orthogonal/spectral norm ≈ 1), normalization, activation functions, structure, or regularization—to ensure that the spectra of each layer are controlled.<br><br>

Summary: After the fact, it is possible to “cut losses and roll back,” but not to “reverse divergence”; to fundamentally resolve divergent terms, one must modify the initialization, normalization, or structure that gives rise to the factor causing the divergence.